# Dimension scaling — the h_min fix and the two discriminating diagnostics

Follow-up to `dimscale.json` (DECISIONS.md section 25.R). Reuses the saved P_θ per d; no P model is
retrained.

1. **d = 128, sweep (a) rerun** with `h_min` from the **0.1th percentile** of L\* instead of half its
   minimum. In the original run that cell was the only one where `h_min` hit the 1e-4 clamp, and h_ψ
   collapsed onto the floor (tower 1e-4, corr nan).
2. **`grad_ratio_curve`** — ‖∇log h‖ / ‖∇log p‖ — for every cell.
3. **The tower property on the sampler's own reverse trajectories**, for every cell.

### What the reverse tower is measured against

Not 1. Since q(x) = p(x)L\*(x), marginalising the forward kernel gives **q_τ(z) = p_τ(z)·h_τ(z)**, so

* on forward-noised P_θ samples, E_{p_τ}[h_τ] = 1 — the ordinary tower;
* on the sampler's own trajectories, whose marginal should be q_τ, the expectation is
  **E_{q_τ}[h_τ] = E_{p_τ}[h_τ²] ≥ 1**, with equality only if h is constant.

Scoring reverse trajectories against 1 would manufacture a failure in a correct sampler. The statistic
reported is therefore the ratio of the measured reverse mean to E_{p_τ}[h_τ²] estimated on forward-noised
samples; **its target is 1**, and a deviation means the sampler is not realising the measure h implies.

Note what this can and cannot see: it uses the same h_ψ to define the target and to measure, so it tests
whether the sampler *realises* h's measure. It is blind to errors in h itself — which is the point, since
the h-value diagnostics already cover those and look healthy.

In [ ]:
PINNED_COMMIT    = "bbf2c182fde874eddc85762ecc5fc59f6e6df53b"
NOTEBOOK_VERSION = "dimfix-2026.09.25b"
EXPECT_TASKC     = "taskc-2026.09.25b"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment

In [ ]:
import os, sys, json, math, time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import taskc
from taskc.config import frozen
from taskc.data import make_loader
from taskc.ptheta import make_schedule, build_model
from taskc.sampler import reverse_ancestral
from taskc.hnet import HNet, train_hnet, tower_curve, h0_vs_L, grad_ratio_curve
from taskc.smt import make_eps_correction
from taskc.gmm import (make_mixture, gm_sample, tilt, mean_qstar, kl_qstar, beta_for_kl,
                       heldout_spec, solve_beta_samples, resample, sliced_w1)

_nb_path = "notebooks/taskc_12_dimscale_hfix_colab.ipynb"
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb_path).read(), (
    f"{NOTEBOOK_VERSION} does not appear in this notebook at PINNED_COMMIT ({PINNED_COMMIT[:7]}). "
    f"Stale cached notebook, or the pin was not advanced with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_dimscale"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/dimscale_local")
print("taskc:", taskc.__version__, "| device:", DEVICE, "| tf32:", torch.backends.cuda.matmul.allow_tf32)
prev = json.load(open(os.path.join(DRIVE, "dimscale.json")))
DIMS, B_FIXED, KL_FIXED = prev["dims"], prev["b_fixed"], prev["kl_fixed"]
N_POOL, N_EVAL, N_HPSI = prev["n_pool"], prev["n_eval"], 100_000
N_EXACT, H_EPOCHS, N_PROJ = 200_000, 100, 8
TS = (0, 100, 300, 600, 900)
N_TRAJ = 20_000            # trajectories for the reverse tower
SKIP_DONE = True

RES = os.path.join(DRIVE, "dimscale_hfix.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__,
    device=DEVICE, of_run=prev.get("notebook"), cells={}, rerun={}, timings={})
def save(): json.dump(res, open(RES, "w"), indent=1, default=float)
save()

def cfg_for(d): return frozen(data_dim=d, artifact_dir=DRIVE, z_cap=1e9)
def betas_for(gm, d):
    return {"a_fixed_tilt": B_FIXED*np.ones(d), "b_fixed_kl": beta_for_kl(gm, np.ones(d), KL_FIXED)}
def h_min_old(L): return float(max(1e-4, 0.5*L.min()))            # the rule that failed
H_FLOOR = 1e-8            # was 1e-4, which clamped the percentile rule and voided the fix
def h_min_new(L): return float(max(H_FLOOR, np.percentile(L, 0.1)))
print("dims", DIMS, "| pool", N_POOL, "| done:", list(res["cells"]))

### Stage 0b — the reverse-trajectory tower

Replicates the frozen reverse chain exactly (same arithmetic as `taskc.sampler.reverse_ancestral`) and
records the state at each checkpoint τ, so h_τ can be evaluated on the sampler's own trajectory.

In [ ]:
@torch.no_grad()
def reverse_with_checkpoints(model, shape, sched, device, eps_correction, ts, seed):
    """Same arithmetic as taskc.sampler.reverse_ancestral, recording y at each t in ts."""
    torch.manual_seed(seed)
    if str(device) == "cuda": torch.cuda.manual_seed_all(seed)
    alphas, abar, betas = sched.alphas, sched.alphas_bar, sched.betas
    B, D = shape; T = abar.shape[0]; t_start = sched.t_start
    want = set(int(t) for t in ts); got = {}
    y = torch.randn(B, D, device=device)
    for t in reversed(range(t_start + 1)):
        if t in want: got[t] = y.clone()
        a_bar_t = abar[t]
        t01 = torch.full((B,), (t + 0.5) / T, device=device)
        eps_hat = model(y, t01)
        if eps_correction is not None:
            eps_hat = eps_hat + eps_correction(y, t01, t)
        if t > 0:
            pv = torch.clamp((1.0 - abar[t-1]) / (1.0 - a_bar_t) * betas[t], min=1e-12)
            mean = (1.0/torch.sqrt(alphas[t])) * (y - (betas[t]/torch.sqrt(1.0 - a_bar_t))*eps_hat)
            y = mean + torch.sqrt(pv) * torch.randn_like(y)
        else:
            y = (1.0/torch.sqrt(alphas[t])) * (y - (betas[t]/torch.sqrt(1.0 - a_bar_t))*eps_hat)
    return y, got

@torch.no_grad()
def reverse_tower(hnet, model, sched, d, ts, z0_fwd, device, seed, n=20_000):
    """Reverse-trajectory mean of h_tau, against its correct target E_{p_tau}[h_tau^2]."""
    delta = make_eps_correction(hnet, sched)
    _, ck = reverse_with_checkpoints(model, (n, d), sched, device, delta, ts, seed)
    abar = sched.alphas_bar.to(device); T = sched.T
    z0 = torch.from_numpy(np.asarray(z0_fwd[:n], dtype=np.float32)).to(device)
    g = torch.Generator().manual_seed(seed + 1)
    out = {}
    for t in ts:
        t01 = torch.full((n,), (t + 0.5) / T, device=device)
        hr = hnet(ck[t], t01, abar[t].expand(n)).cpu().numpy()
        eps = torch.randn(z0.shape, generator=g).to(device)
        zt = torch.sqrt(abar[t]) * z0 + torch.sqrt(1 - abar[t]) * eps
        hf = hnet(zt, t01, abar[t].expand(n)).cpu().numpy()
        pred = float((hf ** 2).mean())                 # E_{p_tau}[h^2] = E_{q_tau}[h]
        meas = float(hr.mean())
        se = float(hr.std(ddof=1) / math.sqrt(n))
        out[int(t)] = dict(measured=meas, predicted=pred, fwd_tower=float(hf.mean()),
                           ratio=(meas / pred if pred > 0 else float("nan")),
                           se=se, z=((meas - pred) / se if se > 0 else float("nan")))
    return out

### Stage 1 — diagnostics for every cell, under the ORIGINAL h_min rule

Retrained with the original rule and the original seeds so the diagnostics describe the h_ψ that
produced `dimscale.json`, not a different one.

In [ ]:
POOLS = {}
def get_pool(d, model, sched):
    if d in POOLS: return POOLS[d]
    cfg = cfg_for(d); out = []
    torch.manual_seed(50_000 + d)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(50_000 + d)
    for lo in range(0, N_POOL, 50_000):
        m = min(50_000, N_POOL - lo)
        x = reverse_ancestral(model, (m, d), sched.alphas, sched.alphas_bar, sched.betas,
                              DEVICE, t_start=sched.t_start, eps_correction=None)
        out.append(x.detach().cpu().numpy().astype(np.float64))
    POOLS[d] = np.concatenate(out, 0); return POOLS[d]

for d in DIMS:
    gm = make_mixture(d, K=4, seed=0); cfg = cfg_for(d)
    sched = make_schedule(cfg, device=DEVICE)
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(DRIVE, f"ptheta_d{d}.pt"),
                                     map_location=DEVICE)["sd"]); model.eval()
    Xpool = get_pool(d, model, sched)
    for sweep, beta_true in betas_for(gm, d).items():
        key = f"d{d}_{sweep}"
        if key in res["cells"]: print(f"{key}: cached"); continue
        t0 = time.time()
        c = mean_qstar(gm, beta_true)
        beta_hat, w_pool, kl_hat, ess = solve_beta_samples(Xpool, c)
        zz = Xpool @ beta_hat
        L_pool = np.exp(zz - (np.log(np.mean(np.exp(zz - zz.max()))) + zz.max()))
        sub = np.random.default_rng(70_000 + d).choice(len(Xpool), size=min(N_HPSI, len(Xpool)),
                                                       replace=False)
        Ls = L_pool[sub]
        hm = h_min_old(Ls)
        hn = HNet(d, 256, 32, hm)
        train_hnet(hn, Xpool[sub], Ls, sched, device=DEVICE, epochs=H_EPOCHS, verbose=False)
        gr = grad_ratio_curve(hn, model, Xpool[sub], sched, TS, device=DEVICE)
        tw = tower_curve(hn, Xpool[sub][:N_TRAJ], sched, TS, device=DEVICE)
        h0 = h0_vs_L(hn, Xpool[sub][:N_TRAJ], Ls[:N_TRAJ], sched, device=DEVICE)
        rt = reverse_tower(hn, model, sched, d, TS, Xpool[sub], DEVICE, seed=81_000 + d, n=N_TRAJ)
        res["cells"][key] = dict(d=d, sweep=sweep, ess=float(ess), h_min=hm,
                                 L_min=float(Ls.min()), L_p01=float(np.percentile(Ls, 0.1)),
                                 grad_ratio=gr, tower_fwd={str(t): v["mean"] for t, v in tw.items()},
                                 h0_vs_L=h0, reverse_tower=rt)
        res["timings"][key] = time.time() - t0; save()
        print(f"{key}: h_min {hm:.2e} (L min {Ls.min():.2e}, p0.1 {np.percentile(Ls,0.1):.2e})  "
              f"corr {h0['corr']:.4f}  grad ratio@300 {gr[300]['ratio_mean']:.4f}  "
              f"rev tower ratio@300 {rt[300]['ratio']:.4f}  ({time.time()-t0:.0f}s)", flush=True)

### Stage 2 — d = 128, sweep (a), rerun with the new h_min

`h_min` from the 0.1th percentile of L\*. Everything else is identical, including the seeds.

In [ ]:
KEY = "d128_a_fixed_tilt"
FIX_KEY = "fix_floor1e-8"     # keyed by the floor so a stale result cannot satisfy SKIP_DONE
if FIX_KEY not in res["rerun"]:
    t0 = time.time(); d = 128
    gm = make_mixture(d, K=4, seed=0); cfg = cfg_for(d)
    sched = make_schedule(cfg, device=DEVICE)
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(DRIVE, f"ptheta_d{d}.pt"),
                                     map_location=DEVICE)["sd"]); model.eval()
    beta_true = betas_for(gm, d)["a_fixed_tilt"]
    q_exact = tilt(gm, beta_true); c = mean_qstar(gm, beta_true)
    spec = heldout_spec(q_exact, n_proj=N_PROJ, seed=77)
    Xq_exact = gm_sample(q_exact, N_EXACT, seed=40_000 + d)
    Xpool = get_pool(d, model, sched)
    beta_hat, w_pool, kl_hat, ess = solve_beta_samples(Xpool, c)
    zz = Xpool @ beta_hat
    L_pool = np.exp(zz - (np.log(np.mean(np.exp(zz - zz.max()))) + zz.max()))
    sub = np.random.default_rng(70_000 + d).choice(len(Xpool), size=min(N_HPSI, len(Xpool)), replace=False)
    Ls = L_pool[sub]
    hm = h_min_new(Ls)
    print(f"h_min: old rule {h_min_old(Ls):.3e} -> new rule {hm:.3e}  "
          f"(L* min {Ls.min():.3e}, p0.1 {np.percentile(Ls,0.1):.3e}, mean {Ls.mean():.4f})")
    hn = HNet(d, 256, 32, hm)
    train_hnet(hn, Xpool[sub], Ls, sched, device=DEVICE, epochs=H_EPOCHS, verbose=False)
    gr = grad_ratio_curve(hn, model, Xpool[sub], sched, TS, device=DEVICE)
    tw = tower_curve(hn, Xpool[sub][:N_TRAJ], sched, TS, device=DEVICE)
    h0 = h0_vs_L(hn, Xpool[sub][:N_TRAJ], Ls[:N_TRAJ], sched, device=DEVICE)
    rt = reverse_tower(hn, model, sched, d, TS, Xpool[sub], DEVICE, seed=81_000 + d, n=N_TRAJ)
    # full SMT metrics, exactly as in the original run
    delta = make_eps_correction(hn, sched); out = []
    torch.manual_seed(80_000 + d)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(80_000 + d)
    t1 = time.time()
    for lo in range(0, N_EVAL, 50_000):
        m = min(50_000, N_EVAL - lo)
        x = reverse_ancestral(model, (m, d), sched.alphas, sched.alphas_bar, sched.betas,
                              DEVICE, t_start=sched.t_start, eps_correction=delta)
        out.append(x.detach().cpu().numpy().astype(np.float64))
    X = np.concatenate(out, 0); smt_s = time.time() - t1
    n = X.shape[0]; se = X.std(0)/math.sqrt(n)
    mz = (X.mean(0) - c)/np.where(se > 0, se, np.inf)
    P = X @ spec["U"].T
    v = P.var(0); tl = (P > spec["thresh"][None, :]).mean(0)
    v_se = np.sqrt(np.maximum(((P - P.mean(0))**4).mean(0) - v**2, 0)/n)
    t_se = np.sqrt(np.maximum(tl*(1-tl), 0)/n)
    frac_floor = {str(t): v["frac_near_floor"] for t, v in tw.items()}
    collapsed = bool(np.mean([v for v in frac_floor.values()]) > 0.9)
    res["rerun"][FIX_KEY] = dict(
        key=KEY, h_floor=H_FLOOR, h_min_old=h_min_old(Ls), h_min_new=hm, ess=float(ess),
        frac_at_floor=frac_floor, collapsed=collapsed,
        grad_ratio=gr, tower_fwd={str(t): vv["mean"] for t, vv in tw.items()},
        h0_vs_L=h0, reverse_tower=rt,
        smt=dict(mean_max_z=float(np.abs(mz).max()), mean_rms_z=float(np.sqrt((mz**2).mean())),
                 var_max_z=float(np.abs((v-spec["var"])/np.where(v_se>0, v_se, np.inf)).max()),
                 tail_max_z=float(np.abs((tl-spec["tail"])/np.where(t_se>0, t_se, np.inf)).max()),
                 sw_to_exact_Q=float(sliced_w1(X, Xq_exact, 256, seed=9)), seconds=smt_s))
    res["timings"][FIX_KEY] = time.time()-t0; save()
F = res["rerun"][FIX_KEY]; O = prev["cells"][KEY]
print(f"\nh floor {F['h_floor']:.0e}; fraction of h at the floor by tau: "
      + ", ".join(f"{t}:{v:.3f}" for t, v in F["frac_at_floor"].items()))
print("VERDICT: h_psi COLLAPSED AGAIN -- with the floor out of the way this is a genuine limit "
      "of the method at this KL/ESS, not a floor artifact." if F["collapsed"] else
      "VERDICT: h_psi did NOT collapse -- the original failure was the 1e-4 floor.")
print(f"\n{'':22s} {'ORIGINAL (old h_min rule)':>26s} {'FIXED (0.1th pct)':>22s}")
print(f"{'h_min':22s} {F['h_min_old']:26.3e} {F['h_min_new']:22.3e}")
print(f"{'tower(0)':22s} {O['tower']['0']:26.4f} {F['tower_fwd']['0']:22.4f}")
print(f"{'corr(h0,L*)':22s} {O['h0_vs_L']['corr']:26.4f} {F['h0_vs_L']['corr']:22.4f}")
for k, lab in (("mean_max_z","SMT mean max|z|"),("tail_max_z","SMT tail max|z|"),
               ("sw_to_exact_Q","SMT SW to exact Q*")):
    print(f"{lab:22s} {O['arms']['SMT'][k]:26.4f} {F['smt'][k]:22.4f}")
print(f"{'(weight+SIR SW was':22s} {prev['cells'][KEY]['arms']['weight_sir']['sw_to_exact_Q']:26.4f})")

### Stage 3 — the two diagnostics across all cells

In [ ]:
print("="*118)
print("grad ratio  ||grad log h|| / ||grad log p||   and   reverse-trajectory tower  measured / E_p[h^2]")
print("="*118)
print(f"{'cell':22s} {'ESS%':>7s} {'h_min':>9s} {'corr':>7s} " +
      "".join(f"{'gr@'+str(t):>9s}" for t in TS) + "  " +
      "".join(f"{'rt@'+str(t):>9s}" for t in TS))
for k, c in res["cells"].items():
    gr = c["grad_ratio"]; rt = c["reverse_tower"]
    print(f"{k:22s} {c['ess']*100:7.2f} {c['h_min']:9.2e} {c['h0_vs_L']['corr']:7.4f} " +
          "".join(f"{gr[str(t)]['ratio_mean'] if str(t) in gr else gr[t]['ratio_mean']:9.2e}" for t in TS) + "  " +
          "".join(f"{rt[str(t)]['ratio'] if str(t) in rt else rt[t]['ratio']:9.4f}" for t in TS))
print("\ngrad ratio: size of the correction relative to the base score.")
print("reverse tower: target 1. Below 1 = the sampler under-realises the tilt h implies;")
print("               above 1 = it over-realises it. Either way the sampler is not producing q_tau.")
print("               Blind to errors in h itself, which the value diagnostics already cover.")
print("\ntimings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `dimscale_hfix.json`. Outcomes go to DECISIONS.md section 25.R.2 as an amendment,
including whether the h_min fix recovers the cell and what the two diagnostics say about the gradient
versus trajectory-mismatch hypotheses.